# 📊 AI-Assisted Trading Risk Manager
**Sentiment → Risk Models → Portfolio Optimisation → Trailing Stops → Monte Carlo**

**Install dependencies:**
```bash
pip install yfinance transformers torch numpy scipy plotly arch hmmlearn
```

> **All tunable parameters live in the two config cells below.**  
> You should rarely need to touch anything else.


## ⚙️ Master Configuration
*Edit anything here — every downstream cell reads from these variables.*

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────────────────────
# 1. PORTFOLIO
# ─────────────────────────────────────────────────────────────────────────────
TICKERS    = ['AAPL', 'MSFT', 'XOM', 'GS', 'JPM']  # tickers to analyse
START      = '2020-01-01'   # historical data start
END        = '2024-12-31'   # historical data end
RISK_FREE  = 0.05           # annual risk-free rate (decimal)

# ─────────────────────────────────────────────────────────────────────────────
# 2. SENTIMENT
# ─────────────────────────────────────────────────────────────────────────────
MAX_HEADLINES_PER_TICKER = 6   # headlines fetched per ticker via yfinance

# ─────────────────────────────────────────────────────────────────────────────
# 3. RISK MODELS
# ─────────────────────────────────────────────────────────────────────────────
VOL_TICKER           = 'AAPL'         # ticker used for vol / regime analysis
ROLLING_WINDOW       = 21             # days for rolling volatility window
EWMA_SPAN            = 21             # span for EWMA (≈ λ = 0.94 at span=21)
GARCH_P              = 1              # GARCH lag order p
GARCH_Q              = 1              # GARCH lag order q
VAR_CONFIDENCE_LEVELS = (0.95, 0.99)  # confidence levels for VaR / CVaR table
VAR_SIM_SIZE         = 100_000        # Monte Carlo draws for parametric VaR

# ─────────────────────────────────────────────────────────────────────────────
# 4. REGIME DETECTION (HMM)
# ─────────────────────────────────────────────────────────────────────────────
HMM_N_REGIMES = 3    # number of hidden market regimes (Bull / Neutral / Bear)
HMM_N_ITER    = 200  # EM iterations for HMM fitting

# ─────────────────────────────────────────────────────────────────────────────
# 5. PORTFOLIO OPTIMISATION
# ─────────────────────────────────────────────────────────────────────────────
N_SIM_EF              = 3_000   # random portfolios for efficient frontier
SENTIMENT_RETURN_BOOST = 0.01   # expected return nudge per unit of sentiment score
                                 # (e.g. 0.01 = +1 % return boost per unit)

# ─────────────────────────────────────────────────────────────────────────────
# 6. MONTE CARLO
# ─────────────────────────────────────────────────────────────────────────────
HORIZON            = 252         # simulation horizon in trading days (~1 year)
N_PATHS            = 10_000      # number of simulated paths
PORT_VAL           = 1_000_000   # starting portfolio value ($)
RANDOM_SEED        = 42          # for reproducibility (set None for truly random)
PLOT_SAMPLE_PATHS  = 200         # paths drawn on the chart (cosmetic, not analytical)
SENTIMENT_DRIFT_NUDGE = 0.0002   # daily drift nudge per unit of portfolio sentiment

# Regime GBM parameters — ANNUAL drift & vol (converted to daily inside the MC cell)
REGIME_PARAMS = {
    'Bull 🟢':    {'drift':  0.12, 'vol': 0.12},
    'Neutral ⚪': {'drift':  0.04, 'vol': 0.18},
    'Bear 🔴':    {'drift': -0.08, 'vol': 0.28},
}

# ─────────────────────────────────────────────────────────────────────────────
# 7. STRESS TEST
# ─────────────────────────────────────────────────────────────────────────────
N_PATHS_STRESS = 5_000   # paths per stress scenario

# Annual drift shock & vol multiplier for each scenario
SCENARIOS = {
    'Oil −20%':          {'drift_shock': -0.06, 'vol_mult': 1.4},
    'Rates +100 bps':    {'drift_shock': -0.03, 'vol_mult': 1.2},
    'Market crash −30%': {'drift_shock': -0.25, 'vol_mult': 2.5},
    'Base case':         {'drift_shock':  0.0,  'vol_mult': 1.0},
}

print('✅ Master config loaded.')


✅ Master config loaded.


## ⚙️ Trailing Stop Configuration
Each position is split into **3 tranches** (⅓ each).  
A trailing stop watches the **drawdown from the running peak**; when breached, that tranche locks in as cash.

**Two modes:**
- `USE_MANUAL = False` *(default)* — levels are **auto-derived from the sentiment score** after Phase 1 runs.  
- `USE_MANUAL = True` — uses your `MANUAL_STOP_LEVELS` / `MANUAL_STOP_FRACTIONS` directly.


In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# TRAILING STOP PARAMETERS
# ─────────────────────────────────────────────────────────────────────────────

USE_MANUAL = False   # ← flip to True to ignore sentiment-derived levels

# ── Manual override (active when USE_MANUAL = True) ───────────────────────
# Drawdown thresholds (negative decimals) and fraction of remaining position to exit.
MANUAL_STOP_LEVELS    = [-0.05, -0.10, -0.15]   # e.g. -0.05 = 5 % drawdown from peak
MANUAL_STOP_FRACTIONS = [1/3,   1/3,   1/3  ]   # must sum to ≤ 1

# ── Sentiment thresholds for regime classification ────────────────────────
SENTIMENT_BULL_THRESHOLD =  0.3   # score ≥ this → bullish stops
SENTIMENT_BEAR_THRESHOLD = -0.3   # score ≤ this → bearish stops

# ── Auto-derived levels per sentiment regime ──────────────────────────────
# Bullish → wider stops (give the position more room to run)
BULL_STOP_LEVELS    = [-0.08, -0.14, -0.20]
BULL_STOP_FRACTIONS = [1/3,   1/3,   1/3  ]

# Neutral → moderate stops
NEUTRAL_STOP_LEVELS    = [-0.05, -0.10, -0.15]
NEUTRAL_STOP_FRACTIONS = [1/3,   1/3,   1/3  ]

# Bearish → tighter stops (protect capital more aggressively)
BEAR_STOP_LEVELS    = [-0.03, -0.06, -0.10]
BEAR_STOP_FRACTIONS = [1/3,   1/3,   1/3  ]

# ── Builder — called after Phase 1 produces portfolio_sentiment ───────────
def build_stops(sentiment_score):
    """Return list of stop dicts based on sentiment score and config above."""
    if USE_MANUAL:
        levels, fracs = MANUAL_STOP_LEVELS, MANUAL_STOP_FRACTIONS
        tag = '⚙️  Manual'
    elif sentiment_score >= SENTIMENT_BULL_THRESHOLD:
        levels, fracs = BULL_STOP_LEVELS, BULL_STOP_FRACTIONS
        tag = '🟢 Bullish-derived'
    elif sentiment_score <= SENTIMENT_BEAR_THRESHOLD:
        levels, fracs = BEAR_STOP_LEVELS, BEAR_STOP_FRACTIONS
        tag = '🔴 Bearish-derived'
    else:
        levels, fracs = NEUTRAL_STOP_LEVELS, NEUTRAL_STOP_FRACTIONS
        tag = '⚪ Neutral-derived'

    stops = [
        {'level': lvl, 'exit_fraction': frac,
         'label': f'Stop {i+1} ({lvl:.0%})'}
        for i, (lvl, frac) in enumerate(zip(levels, fracs))
    ]
    return stops, tag

print('✅ Trailing stop config loaded.  Run Phase 1 to finalise levels.')


✅ Trailing stop config loaded.  Run Phase 1 to finalise levels.


---
## Phase 1 — Financial Sentiment Engine (FinBERT + live news)
Fetches real headlines from Yahoo Finance via `yfinance` for each ticker.  
Falls back to a curated sample set only if the network returns nothing.


In [3]:
from transformers import pipeline

# ── 1a. Fetch real headlines ──────────────────────────────────────────────
def get_ticker_headlines(tickers, max_per=MAX_HEADLINES_PER_TICKER):
    """Fetch news via yfinance. Handles both old and new API response formats."""
    results = []
    for t in tickers:
        try:
            news = yf.Ticker(t).news or []
            for item in news[:max_per]:
                if 'content' in item and isinstance(item['content'], dict):
                    title = item['content'].get('title', '')
                else:
                    title = item.get('title', '')
                if title:
                    results.append({'ticker': t, 'headline': title})
        except Exception as e:
            print(f'  ⚠️  {t}: {e}')
    return results

print('Fetching live headlines...')
live_items = get_ticker_headlines(TICKERS)

FALLBACK_HEADLINES = [
    {'ticker': 'AAPL', 'headline': 'Apple beats earnings expectations, raises guidance'},
    {'ticker': 'AAPL', 'headline': 'iPhone 16 demand stronger than anticipated in Asia'},
    {'ticker': 'MSFT', 'headline': 'Microsoft Azure cloud revenue surges 33% year-on-year'},
    {'ticker': 'MSFT', 'headline': 'Goldman Sachs upgrades Microsoft to strong buy'},
    {'ticker': 'XOM',  'headline': 'Oil prices tumble on demand fears amid global slowdown'},
    {'ticker': 'XOM',  'headline': 'ExxonMobil raises dividend as oil profits remain elevated'},
    {'ticker': 'GS',   'headline': 'Goldman Sachs Q3 profit rises on trading revenue rebound'},
    {'ticker': 'GS',   'headline': 'Wall Street banks face tighter capital requirements'},
    {'ticker': 'JPM',  'headline': 'JPMorgan warns of credit losses in commercial real estate'},
    {'ticker': 'JPM',  'headline': 'Fed signals two more rate hikes this year'},
]

if live_items:
    print(f'✅ {len(live_items)} live headlines fetched.')
    headline_items = live_items
else:
    print('⚠️  No live data — using fallback headlines.')
    headline_items = FALLBACK_HEADLINES

headlines = [h['headline'] for h in headline_items]

# ── 1b. FinBERT ───────────────────────────────────────────────────────────
print('\nLoading FinBERT (~420 MB on first run)...')
sentiment_pipe = pipeline('text-classification', model='ProsusAI/finbert', top_k=None)

results = sentiment_pipe(headlines)
rows = []
for item, scores in zip(headline_items, results):
    if isinstance(scores, dict):
        scores = [scores]
    sd = {s['label']: s['score'] for s in scores}
    rows.append({
        'ticker':    item['ticker'],
        'headline':  item['headline'],
        'positive':  round(sd.get('positive', 0), 3),
        'negative':  round(sd.get('negative', 0), 3),
        'neutral':   round(sd.get('neutral',  0), 3),
        'sentiment': max(sd, key=sd.get),
    })

sentiment_df = pd.DataFrame(rows)
display(sentiment_df)

# ── 1c. Aggregates ────────────────────────────────────────────────────────
ticker_sentiment = (
    sentiment_df.groupby('ticker')
    .apply(lambda g: g['positive'].mean() - g['negative'].mean())
    .rename('score').round(3)
)
portfolio_sentiment = ticker_sentiment.mean()

print('\n📊 Per-ticker sentiment:')
print(ticker_sentiment.to_string())
print(f'\n📰 Portfolio sentiment score: {portfolio_sentiment:+.3f}  (-1 = very bearish, +1 = very bullish)')

# ── 1d. Finalise trailing stops ───────────────────────────────────────────
STOPS, stop_tag = build_stops(portfolio_sentiment)
print(f'\n🎯 Trailing stops ({stop_tag}) | score={portfolio_sentiment:+.3f}:')
print(f'  {"Level":>10}   {"Exit fraction":>14}   Label')
print('  ' + '─'*42)
for s in STOPS:
    print(f'  {s["level"]:>+10.1%}   {s["exit_fraction"]:>14.1%}   {s["label"]}')
print('\nTo override, set USE_MANUAL = True in the config cell and re-run.')


Fetching live headlines...
✅ 30 live headlines fetched.

Loading FinBERT (~420 MB on first run)...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 21131.85it/s]


,ticker,headline,positive,negative,neutral,sentiment
0,AAPL,"This ""Magnificent Seven"" Stock Is the Worst Pe...",0.049,0.456,0.495,neutral
1,AAPL,Is Intel’s (INTC) Confidential AI Push Quietly...,0.031,0.257,0.712,neutral
2,AAPL,Apple Inc. (AAPL)’s Durable Growth Narrative K...,0.933,0.013,0.054,positive
3,AAPL,Why American Express Is Still a Top Buffett St...,0.060,0.025,0.915,neutral
4,AAPL,CDL’s $2.29 annual dividend beats Treasury yie...,0.942,0.029,0.029,positive
5,AAPL,"Trump’s 3,711 Trades Point to Multiple Stock-M...",0.029,0.030,0.940,neutral
6,MSFT,AI trade: Investors may want to look outside i...,0.064,0.015,0.921,neutral
7,MSFT,"This ""Magnificent Seven"" Stock Is the Worst Pe...",0.049,0.456,0.495,neutral
8,MSFT,Microsoft Maia Chip Talks With Anthropic Test ...,0.288,0.008,0.703,neutral
9,MSFT,The 401(k) Mega Backdoor Roth Strategy a Tech ...,0.052,0.017,0.931,neutral



📊 Per-ticker sentiment:
ticker
AAPL    0.206
GS      0.121
JPM    -0.454
MSFT    0.152
XOM     0.294

📰 Portfolio sentiment score: +0.064  (-1 = very bearish, +1 = very bullish)

🎯 Trailing stops (⚪ Neutral-derived) | score=+0.064:
       Level    Exit fraction   Label
  ──────────────────────────────────────────
       -5.0%            33.3%   Stop 1 (-5%)
      -10.0%            33.3%   Stop 2 (-10%)
      -15.0%            33.3%   Stop 3 (-15%)

To override, set USE_MANUAL = True in the config cell and re-run.


---
## Phase 2 — Risk Models
### 2a · Download price data & compute returns

In [4]:
raw    = yf.download(TICKERS, start=START, end=END, auto_adjust=True)['Close']
prices = raw.dropna()
rets   = prices.pct_change().dropna()
print(f'Downloaded {len(prices)} days for {len(TICKERS)} tickers.')
prices.tail(3)


[*********************100%***********************]  5 of 5 completed

Downloaded 1257 days for 5 tickers.


Ticker,AAPL,GS,JPM,MSFT,XOM
Date,,,,,
2024-12-26,257.375549,566.623291,235.823608,432.973633,101.351982
2024-12-27,253.967392,561.700317,233.912903,425.482483,101.342468
2024-12-30,250.598892,559.136353,232.118561,419.849365,100.657196


### 2b · Volatility Forecasting (Rolling, EWMA, GARCH)

In [5]:
from arch import arch_model

r = rets[VOL_TICKER] * 100   # scale for GARCH numerical stability

roll_vol = r.rolling(ROLLING_WINDOW).std() * np.sqrt(252) / 100
ewma_vol = r.ewm(span=EWMA_SPAN).std()     * np.sqrt(252) / 100

garch     = arch_model(r, vol='Garch', p=GARCH_P, q=GARCH_Q, rescale=False)
garch_fit = garch.fit(disp='off')
garch_vol = garch_fit.conditional_volatility * np.sqrt(252) / 100

fig = go.Figure()
fig.add_trace(go.Scatter(x=roll_vol.index, y=roll_vol,  name=f'Rolling {ROLLING_WINDOW}d'))
fig.add_trace(go.Scatter(x=ewma_vol.index, y=ewma_vol,  name=f'EWMA (span={EWMA_SPAN})'))
fig.add_trace(go.Scatter(x=garch_vol.index, y=garch_vol, name=f'GARCH({GARCH_P},{GARCH_Q})'))
fig.update_layout(title=f'{VOL_TICKER} — Annualised Volatility Forecasts',
                  yaxis_tickformat='.0%', template='plotly_dark')
fig.show()
print(f'Latest GARCH vol: {garch_vol.iloc[-1]:.2%}')


Latest GARCH vol: 21.04%


### 2c · VaR & CVaR Engine

In [6]:
from scipy import stats

def compute_risk(returns, confidence_levels=VAR_CONFIDENCE_LEVELS):
    rows = []
    mu, sigma = returns.mean(), returns.std()
    sim = np.random.normal(mu, sigma, VAR_SIM_SIZE)
    for cl in confidence_levels:
        a   = 1 - cl
        hv  = -np.percentile(returns, a*100)
        hcv = -returns[returns <= -hv].mean()
        pv  = -(mu + stats.norm.ppf(a)*sigma)
        pcv = -(mu - sigma*stats.norm.pdf(stats.norm.ppf(a))/a)
        mv  = -np.percentile(sim, a*100)
        mcv = -sim[sim <= -mv].mean()
        rows.append({'Confidence': f'{cl:.0%}',
                     'Hist VaR': f'{hv:.2%}',   'Hist CVaR': f'{hcv:.2%}',
                     'Param VaR': f'{pv:.2%}',  'Param CVaR': f'{pcv:.2%}',
                     'MC VaR': f'{mv:.2%}',     'MC CVaR': f'{mcv:.2%}'})
    return pd.DataFrame(rows)

display(compute_risk(rets[VOL_TICKER]))


,Confidence,Hist VaR,Hist CVaR,Param VaR,Param CVaR,MC VaR,MC CVaR
0,95%,3.01%,4.44%,3.16%,4.00%,3.16%,3.99%
1,99%,5.03%,7.03%,4.53%,5.20%,4.52%,5.16%


### 2d · Regime Detection (Hidden Markov Model)

In [7]:
from hmmlearn.hmm import GaussianHMM

if RANDOM_SEED is not None:
    np.random.seed(RANDOM_SEED)

r_arr = rets[VOL_TICKER].values.reshape(-1, 1)
hmm   = GaussianHMM(n_components=HMM_N_REGIMES, covariance_type='full',
                    n_iter=HMM_N_ITER, random_state=RANDOM_SEED)
hmm.fit(r_arr)
regimes = hmm.predict(r_arr)
means   = {i: hmm.means_[i][0] for i in range(HMM_N_REGIMES)}
ranking = sorted(means, key=means.get)   # ascending mean → Bear first

# Labels generalise to any number of regimes; middle ones are 'Neutral N'
regime_name_map = {}
for idx, state in enumerate(ranking):
    if idx == 0:
        regime_name_map[state] = 'Bear 🔴'
    elif idx == len(ranking) - 1:
        regime_name_map[state] = 'Bull 🟢'
    else:
        regime_name_map[state] = f'Neutral ⚪' if len(ranking) == 3 else f'Neutral {idx} ⚪'

regime_labels = pd.Series([regime_name_map[r] for r in regimes], index=rets.index)

fig = px.scatter(x=rets.index, y=rets[VOL_TICKER], color=regime_labels,
                 color_discrete_map={'Bear 🔴': 'red', 'Neutral ⚪': 'grey', 'Bull 🟢': 'green'},
                 title=f'{VOL_TICKER} Daily Returns — HMM Regime Detection ({HMM_N_REGIMES} regimes)',
                 template='plotly_dark')
fig.update_traces(marker_size=3)
fig.show()

current_regime = regime_labels.iloc[-1]
print(f'Current regime: {current_regime}')


Current regime: Neutral ⚪


---
## Phase 3 — Portfolio Optimisation (Efficient Frontier)

In [8]:
from scipy.optimize import minimize

mu_annual  = rets.mean() * 252
cov_annual = rets.cov()  * 252
n          = len(TICKERS)

for t in TICKERS:
    if t in ticker_sentiment.index:
        mu_annual[t] += ticker_sentiment[t] * SENTIMENT_RETURN_BOOST

def portfolio_stats(w):
    ret    = w @ mu_annual
    vol    = np.sqrt(w @ cov_annual @ w)
    sharpe = (ret - RISK_FREE) / vol
    return ret, vol, sharpe

def neg_sharpe(w): return -portfolio_stats(w)[2]

constraints = {'type': 'eq', 'fun': lambda w: w.sum() - 1}
bounds      = [(0, 1)] * n
opt         = minimize(neg_sharpe, np.ones(n)/n, method='SLSQP',
                       bounds=bounds, constraints=constraints)
w_sharpe    = opt.x

sim_w     = np.random.dirichlet(np.ones(n), N_SIM_EF)
sim_stats = np.array([portfolio_stats(w) for w in sim_w])

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=sim_stats[:, 1], y=sim_stats[:, 0], mode='markers',
    marker=dict(color=sim_stats[:, 2], colorscale='Viridis', size=4,
                showscale=True, colorbar=dict(title='Sharpe')),
    name=f'{N_SIM_EF} random portfolios'))
r_opt, v_opt, s_opt = portfolio_stats(w_sharpe)
fig.add_trace(go.Scatter(
    x=[v_opt], y=[r_opt], mode='markers+text',
    marker=dict(color='red', size=14, symbol='star'),
    text=['Max Sharpe'], textposition='top right', name='Max Sharpe'))
fig.update_layout(title='Efficient Frontier (sentiment-enhanced returns)',
                  xaxis_title='Volatility', yaxis_title='Return',
                  xaxis_tickformat='.0%', yaxis_tickformat='.0%',
                  template='plotly_dark')
fig.show()

print('\n📌 Optimal weights (Max-Sharpe):')
for t, w in zip(TICKERS, w_sharpe):
    print(f'  {t}: {w:.1%}')
print(f'\n  Return: {r_opt:.2%}  |  Vol: {v_opt:.2%}  |  Sharpe: {s_opt:.2f}')



📌 Optimal weights (Max-Sharpe):
  AAPL: 58.6%
  MSFT: 27.8%
  XOM: 0.0%
  GS: 7.2%
  JPM: 6.3%

  Return: 27.95%  |  Vol: 26.94%  |  Sharpe: 0.85


---
## Phase 4 — Regime-Aware Monte Carlo with Trailing Stops

Each path is simulated via GBM; the **3-tranche trailing stop engine** runs day-by-day:
- Tracks the **running peak** of the in-market portion.
- Drawdown breach → that tranche locks in as cash; the rest keeps running.

Chart: paths coloured by last stop triggered · percentile bands with/without stops · average trigger-day lines.


In [9]:
if RANDOM_SEED is not None:
    np.random.seed(RANDOM_SEED)

# ── Daily regime drift & vol from annual config ───────────────────────────
rp     = REGIME_PARAMS.get(current_regime, REGIME_PARAMS.get('Neutral ⚪',
         list(REGIME_PARAMS.values())[1]))
drift  = rp['drift'] / 252          + portfolio_sentiment * SENTIMENT_DRIFT_NUDGE
vol    = rp['vol']   / np.sqrt(252)

# ── GBM paths ─────────────────────────────────────────────────────────────
shocks     = np.random.normal(0, 1, (HORIZON, N_PATHS))
log_ret    = (drift - 0.5*vol**2) + vol*shocks
daily_rets = np.exp(log_ret)

paths_raw = np.vstack([
    np.full(N_PATHS, float(PORT_VAL)),
    PORT_VAL * np.exp(np.cumsum(log_ret, axis=0))
])

# ── Trailing stop engine ──────────────────────────────────────────────────
stops_sorted = sorted(STOPS, key=lambda s: s['level'])

in_market = np.full(N_PATHS, float(PORT_VAL))
cash      = np.zeros(N_PATHS)
peak      = np.full(N_PATHS, float(PORT_VAL))

triggered   = [np.zeros(N_PATHS, dtype=bool) for _ in stops_sorted]
trigger_day = [np.full(N_PATHS, -1)          for _ in stops_sorted]

paths_stopped    = np.zeros((HORIZON+1, N_PATHS))
paths_stopped[0] = PORT_VAL

for day in range(1, HORIZON+1):
    in_market *= daily_rets[day-1]
    peak       = np.maximum(peak, in_market)
    drawdown   = np.where(peak > 0, (in_market - peak) / peak, 0.0)

    for i, stop in enumerate(stops_sorted):
        fire = (~triggered[i]) & (drawdown <= stop['level'])
        if fire.any():
            locked          = stop['exit_fraction'] * in_market[fire]
            cash[fire]     += locked
            in_market[fire]-= locked
            triggered[i][fire]   = True
            trigger_day[i][fire] = day

    paths_stopped[day] = in_market + cash

# ── Stats ─────────────────────────────────────────────────────────────────
final_raw     = paths_raw[-1]
final_stopped = paths_stopped[-1]
var95_raw     = np.percentile(final_raw,     5)
var95_stopped = np.percentile(final_stopped, 5)
days_ax       = np.arange(HORIZON + 1)
pct           = lambda arr, q: np.percentile(arr, q, axis=1)

# ── Colour paths by last stop triggered ───────────────────────────────────
last_stop    = np.zeros(N_PATHS, dtype=int)
for i in range(len(stops_sorted)):
    last_stop[triggered[i]] = i + 1

STOP_COLOURS = {0: 'steelblue', 1: '#f0e68c', 2: '#ffa500', 3: '#ff4444'}
STOP_NAMES   = {0: 'No stop hit',
                **{i+1: stops_sorted[i]['label'] for i in range(len(stops_sorted))}}

fig = go.Figure()

sample_idx    = np.random.choice(N_PATHS, PLOT_SAMPLE_PATHS, replace=False)
already_shown = set()
for i in sample_idx:
    grp  = last_stop[i]
    name = STOP_NAMES[grp]
    show = name not in already_shown
    already_shown.add(name)
    fig.add_trace(go.Scatter(
        x=days_ax, y=paths_stopped[:, i],
        line=dict(width=0.5, color=STOP_COLOURS[grp]),
        name=name, legendgroup=name, showlegend=show, opacity=0.5))

for q, col, dash, lbl in [(95,'lime','dot','95th (w/ stops)'),
                           (50,'white','solid','Median (w/ stops)'),
                           (5,'red','dot','5th (w/ stops)')]:
    fig.add_trace(go.Scatter(x=days_ax, y=pct(paths_stopped, q),
                             line=dict(color=col, width=2, dash=dash), name=lbl))

for q, lbl in [(50,'Median (no stops)'), (5,'5th (no stops)')]:
    fig.add_trace(go.Scatter(x=days_ax, y=pct(paths_raw, q),
                             line=dict(color='grey', width=1.5, dash='dash'), name=lbl))

for i, stop in enumerate(stops_sorted):
    fired = triggered[i]
    if fired.any():
        avg_day = trigger_day[i][fired].mean()
        fig.add_vline(x=avg_day, line_dash='dot',
                      line_color=STOP_COLOURS[i+1], line_width=1.5,
                      annotation_text=f"{stop['label']}<br>avg day {avg_day:.0f} ({fired.mean():.0%})",
                      annotation_font_size=10)

fig.update_layout(
    title=(f'Monte Carlo — {N_PATHS:,} paths | Regime: {current_regime} '
           f'| Sentiment: {portfolio_sentiment:+.3f}'),
    xaxis_title='Trading Days', yaxis_title='Portfolio Value ($)',
    template='plotly_dark', height=600)
fig.show()

print(f'\n📉 1-Year Risk Summary (regime: {current_regime})')
print(f'  {"":30}  {"No stops":>14}  {"With stops":>14}')
print('  ' + '─'*62)
for label, q in [('Median final value', 50), ('95th pct', 95), ('5th pct', 5)]:
    print(f'  {label:30}  ${np.percentile(final_raw,q):>12,.0f}  '
          f'${np.percentile(final_stopped,q):>12,.0f}')
print(f'  {"VaR 95%":30}  '
      f'${PORT_VAL-var95_raw:>12,.0f}  ${PORT_VAL-var95_stopped:>12,.0f}')

print('\n📊 Trailing stop trigger rates:')
for i, stop in enumerate(stops_sorted):
    fired = triggered[i]
    avg_d = trigger_day[i][fired].mean() if fired.any() else float('nan')
    status = f'triggered in {fired.mean():.1%} of paths | avg day {avg_d:.0f}' if fired.any() else 'never triggered'
    print(f"  {stop['label']:20}  {status}")



📉 1-Year Risk Summary (regime: Neutral ⚪)
                                        No stops      With stops
  ──────────────────────────────────────────────────────────────
  Median final value              $   1,026,811  $   1,002,527
  95th pct                        $   1,377,291  $   1,176,874
  5th pct                         $     764,171  $     902,959
  VaR 95%                         $     235,829  $      97,041

📊 Trailing stop trigger rates:
  Stop 3 (-15%)         triggered in 100.0% of paths | avg day 34
  Stop 2 (-10%)         triggered in 100.0% of paths | avg day 34
  Stop 1 (-5%)          triggered in 100.0% of paths | avg day 33


---
## Stress Test — with trailing stops applied

In [10]:
rows = []
for name, params in SCENARIOS.items():
    d  = drift + params['drift_shock'] / 252
    v  = vol   * params['vol_mult']
    lr = (d - 0.5*v**2) + v * np.random.normal(0, 1, (HORIZON, N_PATHS_STRESS))
    dr = np.exp(lr)

    im   = np.full(N_PATHS_STRESS, float(PORT_VAL))
    cs   = np.zeros(N_PATHS_STRESS)
    pk   = np.full(N_PATHS_STRESS, float(PORT_VAL))
    trig = [np.zeros(N_PATHS_STRESS, dtype=bool) for _ in stops_sorted]

    for day in range(HORIZON):
        im *= dr[day]
        pk  = np.maximum(pk, im)
        dd  = np.where(pk > 0, (im - pk) / pk, 0.0)
        for i, stop in enumerate(stops_sorted):
            fire = (~trig[i]) & (dd <= stop['level'])
            if fire.any():
                locked   = stop['exit_fraction'] * im[fire]
                cs[fire] += locked
                im[fire] -= locked
                trig[i][fire] = True

    f = im + cs
    row = {
        'Scenario':     name,
        'Median P&L':   f'${np.median(f) - PORT_VAL:>+,.0f}',
        '5th pct P&L':  f'${np.percentile(f, 5) - PORT_VAL:>+,.0f}',
        'Prob of loss': f'{(f < PORT_VAL).mean():.1%}',
    }
    for i, stop in enumerate(stops_sorted):
        row[stop['label']] = f'{trig[i].mean():.1%}'
    rows.append(row)

display(pd.DataFrame(rows))


,Scenario,Median P&L,5th pct P&L,Prob of loss,Stop 3 (-15%),Stop 2 (-10%),Stop 1 (-5%)
0,Oil −20%,"$-21,321","$-138,276",59.1%,100.0%,100.0%,100.0%
1,Rates +100 bps,"$-8,368","$-119,248",54.1%,100.0%,100.0%,100.0%
2,Market crash −30%,"$-87,079","$-228,237",74.4%,100.0%,100.0%,100.0%
3,Base case,"$+3,289","$-95,402",48.6%,100.0%,100.0%,100.0%
